# Clase 3 · Notebook 01
# El perceptrón desde cero y las funciones de activación

**Introducción al Deep Learning — Módulo 2, Unidad 1: Redes de neuronas**

Antes de usar librerías, vamos a entender la pieza más básica de una red: la **neurona artificial**.
Construiremos un **perceptrón** con NumPy, lo entrenaremos y veremos en qué consiste su gran limitación.
También dibujaremos las **funciones de activación** más usadas.

Contenido:
1. La neurona artificial: suma ponderada + función de activación.
2. Entrenar un perceptrón para la función lógica AND.
3. La limitación: el problema XOR.
4. Las funciones de activación (lineal, ReLU, Leaky ReLU, sigmoide, tanh).

> 💡 Ejecuta las celdas en orden con **Shift + Enter**.

## 1. La neurona artificial

Una neurona calcula una **suma ponderada** de sus entradas (cada entrada `xᵢ` por su peso `wᵢ`),
le añade un **bias** `b` y aplica una **función de activación** `f`:

$$ y = f\\left(\\sum_i w_i x_i + b\\right) $$

Empezamos con la función **escalón** (step) del perceptrón original: devuelve 1 si la suma supera 0, y 0 si no.

In [ ]:
import numpy as np

def escalon(z):
    return np.where(z >= 0, 1, 0)

def neurona(x, w, b):
    z = np.dot(x, w) + b          # suma ponderada + bias
    return escalon(z)

# Ejemplo: dos entradas, pesos y bias fijos
x = np.array([1, 0])
w = np.array([0.5, 0.5])
b = -0.7
print("Entrada:", x, "-> salida:", neurona(x, w, b))

## 2. Entrenar un perceptrón: la función AND

Vamos a que el perceptrón **aprenda** la función lógica **AND** (solo es 1 si ambas entradas son 1).
El entrenamiento ajusta los pesos según el **error** (regla del perceptrón):

`w ← w + η · (y_real − y_predicho) · x`

In [ ]:
# Datos de la función AND
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])

def entrenar_perceptron(X, y, epocas=20, eta=0.1):
    w = np.zeros(X.shape[1])
    b = 0.0
    for _ in range(epocas):
        for xi, yi in zip(X, y):
            pred = escalon(np.dot(xi, w) + b)
            error = yi - pred
            w = w + eta * error * xi      # actualización de pesos
            b = b + eta * error           # actualización del bias
    return w, b

w, b = entrenar_perceptron(X, y_and)
print("Pesos aprendidos:", w, "| bias:", round(b, 2))
print("\nComprobación (AND):")
for xi, yi in zip(X, y_and):
    print(f"  {xi} -> predicho {escalon(np.dot(xi, w) + b)} (real {yi})")

El perceptrón aprende AND perfectamente, porque las clases son **linealmente separables**: una sola recta las separa.

## 3. La limitación: el problema XOR

La función **XOR** vale 1 cuando las entradas son **distintas**. No existe una sola recta que separe sus clases,
así que un perceptrón simple **no puede aprenderla**. Comprobémoslo.

In [ ]:
y_xor = np.array([0, 1, 1, 0])

w, b = entrenar_perceptron(X, y_xor, epocas=100)
print("Intento de aprender XOR con un perceptrón simple:")
aciertos = 0
for xi, yi in zip(X, y_xor):
    pred = escalon(np.dot(xi, w) + b)
    aciertos += (pred == yi)
    print(f"  {xi} -> predicho {pred} (real {yi})")
print(f"\nAciertos: {aciertos}/4  -> el perceptrón simple NO resuelve el XOR.")

In [ ]:
# Visualicemos por qué: AND es separable con una recta; XOR no
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, y, titulo in [(axes[0], y_and, "AND (separable)"), (axes[1], y_xor, "XOR (no separable)")]:
    for xi, yi in zip(X, y):
        ax.scatter(*xi, c=("#FF647E" if yi else "#000F9F"), s=200, edgecolors="k", zorder=3)
    ax.set_title(titulo); ax.set_xlabel("x1"); ax.set_ylabel("x2")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.grid(alpha=0.3)
axes[0].plot([-0.2, 1.2], [1.3, -0.3], "g--", lw=2)   # recta que separa AND
plt.tight_layout(); plt.show()

La solución es combinar varias neuronas en un **perceptrón multicapa (MLP)**, que sí resuelve el XOR.
Lo construiremos en el **Notebook 02** con Keras.

## 4. Funciones de activación

Las redes profundas necesitan funciones de activación **no lineales** (la función escalón no sirve porque su
derivada es 0). Estas son las más habituales.

In [ ]:
def lineal(z):     return z
def relu(z):       return np.maximum(0, z)
def leaky_relu(z): return np.where(z >= 0, z, 0.1 * z)
def sigmoide(z):   return 1 / (1 + np.exp(-z))
def tanh(z):       return np.tanh(z)

z = np.linspace(-5, 5, 200)
funcs = [("Lineal", lineal), ("ReLU", relu), ("Leaky ReLU", leaky_relu),
         ("Sigmoide", sigmoide), ("Tanh", tanh)]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, (nombre, f) in zip(axes, funcs):
    ax.plot(z, f(z), color="#000F9F", lw=2)
    ax.axhline(0, color="gray", lw=0.6); ax.axvline(0, color="gray", lw=0.6)
    ax.set_title(nombre); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Observa los rangos: ReLU ≥ 0, sigmoide entre 0 y 1, tanh entre -1 y 1.
La elección de la activación afecta a cómo y cómo de rápido aprende la red.

## Resumen

- Una neurona = **suma ponderada + bias + función de activación**.
- Un **perceptrón** aprende ajustando pesos según el error; resuelve problemas **linealmente separables** (AND).
- **No** resuelve el **XOR**: por eso necesitamos redes **multicapa**.
- Las funciones de activación **no lineales** (ReLU, sigmoide, tanh…) son las que dan poder a las redes profundas.

➡️ En el **Notebook 02** construiremos un MLP con Keras que sí aprende el XOR.